In [1]:
import os
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

import spacy
import re

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, os.path.abspath('..'))
from preprocess import clean_text, map_rating_to_sentiment, preprocess_dataframe
from evaluate import print_metrics, plot_confusion_matrix, plot_class_distribution, plot_top_features, compare_algorithms

print('All libraries loaded successfully.')
print(f'scikit-learn version: {__import__("sklearn").__version__}')
print(f'spaCy version: {spacy.__version__}')
print(f'pandas version: {pd.__version__}')

INFO: spaCy model loaded successfully.


All libraries loaded successfully.
scikit-learn version: 1.3.2
spaCy version: 3.7.2
pandas version: 2.1.4


In [2]:
DATA_PATH = os.path.join('data', 'raw', 'reviews.csv')

df_raw = pd.read_csv(DATA_PATH, nrows=50000)

print(f'Dataset shape: {df_raw.shape}')
print(f'\nColumn names: {df_raw.columns.tolist()}')
print(f'\nFirst 3 rows:')
df_raw.head(3)

Dataset shape: (50000, 10)

Column names: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']

First 3 rows:


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...


In [3]:
print('Data Types and Non-Null Counts:')
print(df_raw.info())
print(f'\nMissing values per column:')
print(df_raw.isnull().sum())

Data Types and Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Id                      50000 non-null  int64 
 1   ProductId               50000 non-null  object
 2   UserId                  50000 non-null  object
 3   ProfileName             49995 non-null  object
 4   HelpfulnessNumerator    50000 non-null  int64 
 5   HelpfulnessDenominator  50000 non-null  int64 
 6   Score                   50000 non-null  int64 
 7   Time                    50000 non-null  int64 
 8   Summary                 49998 non-null  object
 9   Text                    50000 non-null  object
dtypes: int64(5), object(5)
memory usage: 3.8+ MB
None

Missing values per column:
Id                        0
ProductId                 0
UserId                    0
ProfileName               5
HelpfulnessNumerator      0
HelpfulnessD

In [4]:
for rating in [1, 3, 5]:
    sample = df_raw[df_raw['Score'] == rating]['Text'].dropna().iloc[0]
    label = map_rating_to_sentiment(rating)
    print(f'Rating {rating} ({label}):')
    print(f'  "{sample[:200]}"')
    print()

Rating 1 (Negative):
  "Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor intended to represent the product as "Jumbo"."

Rating 3 (Neutral):
  "This seems a little more wholesome than some of the supermarket brands, but it is somewhat mushy and doesn't have quite as much flavor either.  It didn't pass muster with my kids, so I probably won't "

Rating 5 (Positive):
  "I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador"



In [5]:
df_raw_clean = df_raw.dropna(subset=['Text', 'Score']).copy()
df_raw_clean['sentiment'] = df_raw_clean['Score'].astype(int).apply(map_rating_to_sentiment)

sentiment_counts = df_raw_clean['sentiment'].value_counts()
print('Sentiment class distribution:')
for sent, count in sentiment_counts.items():
    pct = count / len(df_raw_clean) * 100
    print(f'  {sent}: {count:,} ({pct:.1f}%)')

Sentiment class distribution:
  Positive: 38,418 (76.8%)
  Negative: 7,535 (15.1%)
  Neutral: 4,047 (8.1%)


In [6]:
fig = plot_class_distribution(df_raw_clean['sentiment'])
plt.show()

In [7]:
df_raw_clean['review_length'] = df_raw_clean['Text'].str.split().str.len()

print('Review length statistics (in words):')
print(df_raw_clean.groupby('sentiment')['review_length'].describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
colors = {'Negative': '#d73027', 'Neutral': '#fee090', 'Positive': '#1a9641'}

for ax, sentiment in zip(axes, ['Negative', 'Neutral', 'Positive']):
    data = df_raw_clean[df_raw_clean['sentiment'] == sentiment]['review_length']
    data_clipped = data[data <= 300]
    ax.hist(data_clipped, bins=40, color=colors[sentiment], edgecolor='white', alpha=0.85)
    ax.set_title(f'{sentiment} Reviews', fontweight='bold')
    ax.set_xlabel('Review Length (words)')
    ax.set_ylabel('Count')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5, label=f'Median: {data.median():.0f}')
    ax.legend(fontsize=9)

plt.suptitle('Review Length Distribution by Sentiment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Review length statistics (in words):
             count  mean   std   min   25%   50%    75%     max
sentiment                                                      
Negative    7535.0  88.3  81.3   7.0  39.0  66.0  110.0  1751.0
Neutral     4047.0  94.7  79.7  11.0  42.0  72.0  121.0   845.0
Positive   38418.0  76.0  73.4   6.0  32.0  53.0   92.0  1513.0


In [8]:
raw_example = "  This is an AMAZING product!!! <br> Check it at https://example.com - batteries included!"
print('PREPROCESSING PIPELINE DEMONSTRATION')
print('='*60)
print(f'Original:  {raw_example}')
print(f'Cleaned:   {clean_text(raw_example)}')
print()

negative_example = "Terrible quality. I am returning this garbage immediately."
print(f'Original:  {negative_example}')
print(f'Cleaned:   {clean_text(negative_example)}')

PREPROCESSING PIPELINE DEMONSTRATION
Original:    This is an AMAZING product!!! <br> Check it at https://example.com - batteries included!
Cleaned:   amazing product check battery include

Original:  Terrible quality. I am returning this garbage immediately.
Cleaned:   terrible quality return garbage immediately


In [9]:
print('Applying preprocessing pipeline to full dataset...')
print('This may take several minutes. Please wait.')

df_processed = preprocess_dataframe(
    df_raw_clean,
    text_col='Text',
    rating_col='Score'
)

print(f'\nProcessing complete.')
print(f'Processed dataset shape: {df_processed.shape}')
print(f'\nSample processed text:')
df_processed.head(5)

INFO: Starting preprocessing. Shape: (50000, 12)
INFO: After dropping NaN rows. Shape: (50000, 12)
INFO: Applying text cleaning pipeline. This may take several minutes...


Applying preprocessing pipeline to full dataset...
This may take several minutes. Please wait.


INFO: Preprocessing complete. Final shape: (50000, 13)



Processing complete.
Processed dataset shape: (50000, 2)

Sample processed text:


,clean_text,sentiment
0,buy vitality can dog food product find good qu...,Positive
1,product arrive label jumbo salt peanut peanut ...,Negative
2,confection century light pillowy citrus gelati...,Positive
3,look secret ingredient robitussin believe find...,Negative
4,great taffy great price wide assortment yummy ...,Positive


In [10]:
os.makedirs(os.path.join('data', 'processed'), exist_ok=True)
processed_path = os.path.join('data', 'processed', 'processed_reviews.csv')
df_processed.to_csv(processed_path, index=False)
print(f'Processed data saved to: {processed_path}')

Processed data saved to: data\processed\processed_reviews.csv


In [11]:
X = df_processed['clean_text'].values
y = df_processed['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set size:  {len(X_train):,}')
print(f'Test set size:      {len(X_test):,}')
print(f'\nClass distribution in training set:')
for label, count in zip(*np.unique(y_train, return_counts=True)):
    print(f'  {label}: {count:,} ({count/len(y_train)*100:.1f}%)')

Training set size:  40,000
Test set size:      10,000

Class distribution in training set:
  Negative: 6,028 (15.1%)
  Neutral: 3,238 (8.1%)
  Positive: 30,734 (76.8%)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

mini_corpus = [
    'product quality excellent fast delivery',
    'terrible quality return garbage',
    'okay product nothing special'
]

demo_tfidf = TfidfVectorizer()
demo_matrix = demo_tfidf.fit_transform(mini_corpus)

tfidf_demo_df = pd.DataFrame(
    demo_matrix.toarray().round(3),
    columns=demo_tfidf.get_feature_names_out(),
    index=['Positive Review', 'Negative Review', 'Neutral Review']
)
print('TF-IDF Matrix (small demo):')
tfidf_demo_df

TF-IDF Matrix (small demo):


,delivery,excellent,fast,garbage,nothing,okay,product,quality,return,special,terrible
Positive Review,0.49,0.49,0.49,0.000,0.000,0.000,0.373,0.373,0.000,0.000,0.000
Negative Review,0.00,0.00,0.00,0.529,0.000,0.000,0.000,0.402,0.529,0.000,0.529
Neutral Review,0.00,0.00,0.00,0.000,0.529,0.529,0.402,0.000,0.000,0.529,0.000


In [13]:
algorithms = {
    'Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)),
        ('clf', MultinomialNB(alpha=0.1))
    ]),
    'Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True, min_df=2)),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42))
    ]),
    'Linear SVM': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True, min_df=2)),
        ('clf', LinearSVC(C=1.0, max_iter=2000, random_state=42))
    ])
}

results = {}

for name, pipeline in algorithms.items():
    print(f'Training {name}...')
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'f1': f1, 'accuracy': acc, 'model': pipeline, 'y_pred': y_pred}
    print(f'  Accuracy: {acc:.4f} | Weighted F1: {f1:.4f}')
    print()

print('All models trained.')

Training Naive Bayes...
  Accuracy: 0.8371 | Weighted F1: 0.8030

Training Logistic Regression...
  Accuracy: 0.8536 | Weighted F1: 0.8246

Training Linear SVM...
  Accuracy: 0.8623 | Weighted F1: 0.8458

All models trained.


In [14]:
for name, result in results.items():
    print(f'\n{"="*60}')
    print(f'Model: {name}')
    print(f'{"="*60}')
    print_metrics(y_test, result['y_pred'])


Model: Naive Bayes

Overall Accuracy: 0.8371 (83.71%)

Per-Class Metrics:
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative       0.81      0.49      0.61      1507
     Neutral       0.53      0.09      0.15       809
    Positive       0.85      0.98      0.91      7684

    accuracy                           0.84     10000
   macro avg       0.73      0.52      0.56     10000
weighted avg       0.81      0.84      0.80     10000


Model: Logistic Regression

Overall Accuracy: 0.8536 (85.36%)

Per-Class Metrics:
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative       0.78      0.61      0.69      1507
     Neutral       0.54      0.10      0.16       809
    Positive       0.87      0.98      0.92      7684

    accuracy                           0.85     10000
   macro avg       0.73      0.56      0.59     10000
weighted avg 

In [15]:
f1_scores = {name: res['f1'] for name, res in results.items()}
fig = compare_algorithms(f1_scores)
plt.show()

In [16]:
best_model_name = max(results, key=lambda k: results[k]['f1'])
print(f'Best model: {best_model_name}')

fig = plot_confusion_matrix(y_test, results[best_model_name]['y_pred'])
plt.show()

Best model: Linear SVM


In [17]:
if best_model_name == 'Logistic Regression':
    fig = plot_top_features(results['Logistic Regression']['model'], n=15)
    plt.show()

In [18]:
lr_pipeline = results['Logistic Regression']['model']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(lr_pipeline, X, y, cv=cv, scoring='f1_weighted', n_jobs=-1)
print(f'5-Fold Cross-Validation Weighted F1 Scores:')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

5-Fold Cross-Validation Weighted F1 Scores:
  Fold 1: 0.8195
  Fold 2: 0.8255
  Fold 3: 0.8208
  Fold 4: 0.8276
  Fold 5: 0.8218
Mean: 0.8231 (+/- 0.0030)


In [19]:
sample_idx = np.random.choice(len(X_train), size=min(20000, len(X_train)), replace=False)
X_sample = X_train[sample_idx]
y_sample = y_train[sample_idx]

base_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True, min_df=2)),
    ('clf', LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42))
])

param_grid = {
    'tfidf__max_features': [30000, 50000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.1, 1.0, 10.0]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    base_pipeline, param_grid,
    cv=cv, scoring='f1_weighted',
    n_jobs=-1, verbose=1
)

print('Running grid search...')
grid_search.fit(X_sample, y_sample)

print(f'\nBest Parameters:')
for param, value in grid_search.best_params_.items():
    print(f'  {param}: {value}')
print(f'\nBest CV F1 Score: {grid_search.best_score_:.4f}')

Running grid search...
Fitting 3 folds for each of 12 candidates, totalling 36 fits

Best Parameters:
  clf__C: 10.0
  tfidf__max_features: 30000
  tfidf__ngram_range: (1, 2)

Best CV F1 Score: 0.8201


In [20]:
best_pipeline = grid_search.best_estimator_
best_pipeline.fit(X_train, y_train)

y_pred_best = best_pipeline.predict(X_test)
print('Final tuned model performance on test set:')
print_metrics(y_test, y_pred_best)

fig = plot_confusion_matrix(y_test, y_pred_best)
plt.show()

Final tuned model performance on test set:

Overall Accuracy: 0.8593 (85.93%)

Per-Class Metrics:
------------------------------------------------------------
              precision    recall  f1-score   support

    Negative       0.75      0.67      0.71      1507
     Neutral       0.47      0.25      0.33       809
    Positive       0.90      0.96      0.93      7684

    accuracy                           0.86     10000
   macro avg       0.70      0.63      0.65     10000
weighted avg       0.84      0.86      0.85     10000



In [21]:
import pickle

MODEL_PATH = os.path.join('models', 'sentiment_model.pkl')
os.makedirs(os.path.join('models'), exist_ok=True)

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(best_pipeline, f)

file_size = os.path.getsize(MODEL_PATH) / (1024 * 1024)
print(f'Model saved to: {MODEL_PATH}')
print(f'File size: {file_size:.2f} MB')

Model saved to: models\sentiment_model.pkl
File size: 10.28 MB


In [22]:
with open(MODEL_PATH, 'rb') as f:
    loaded_model = pickle.load(f)

print('Model loaded successfully from disk.')
test_review = 'The product is fantastic and arrived very quickly. Highly recommended!'
prediction = loaded_model.predict([test_review])[0]
print(f'Test review: "{test_review}"')
print(f'Predicted sentiment: {prediction}')

Model loaded successfully from disk.
Test review: "The product is fantastic and arrived very quickly. Highly recommended!"
Predicted sentiment: Positive


In [23]:
def predict_sentiment(review_text: str, model) -> dict:
    cleaned = clean_text(review_text)
    prediction = model.predict([cleaned])[0]
    result = {'review': review_text, 'sentiment': prediction}

    try:
        proba = model.predict_proba([cleaned])[0]
        classes = model.classes_
        result['confidence'] = dict(zip(classes, proba.round(3)))
    except AttributeError:
        pass

    return result


test_reviews = [
    'Absolutely love this product! Best purchase I have made in years.',
    'Worst product ever. Broke after one day. Complete waste of money.',
    'It works fine. Not great, not terrible. Does what it says it does.',
    'Good quality but the delivery was very late. Mixed feelings.',
    'Cannot believe how bad this is. Do not buy this garbage.',
    'Decent product for the price. I would buy it again if needed.',
]

print('INFERENCE RESULTS')
print('='*70)

for review in test_reviews:
    result = predict_sentiment(review, best_pipeline)
    print(f'Review:    "{review[:70]}..."' if len(review) > 70 else f'Review:    "{review}"')
    print(f'Sentiment: {result["sentiment"]}')
    if 'confidence' in result:
        conf_str = ', '.join([f'{k}: {v:.2f}' for k, v in result['confidence'].items()])
        print(f'Confidence: [{conf_str}]')
    print('-'*70)

INFERENCE RESULTS
Review:    "Absolutely love this product! Best purchase I have made in years."
Sentiment: Positive
Confidence: [Negative: 0.00, Neutral: 0.00, Positive: 1.00]
----------------------------------------------------------------------
Review:    "Worst product ever. Broke after one day. Complete waste of money."
Sentiment: Negative
Confidence: [Negative: 1.00, Neutral: 0.00, Positive: 0.00]
----------------------------------------------------------------------
Review:    "It works fine. Not great, not terrible. Does what it says it does."
Sentiment: Negative
Confidence: [Negative: 0.57, Neutral: 0.25, Positive: 0.18]
----------------------------------------------------------------------
Review:    "Good quality but the delivery was very late. Mixed feelings."
Sentiment: Positive
Confidence: [Negative: 0.01, Neutral: 0.11, Positive: 0.89]
----------------------------------------------------------------------
Review:    "Cannot believe how bad this is. Do not buy this garbag